In [1]:
import joblib
import numpy as np
import pandas as pd
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.linear_model import SGDClassifier, SGDRegressor, LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.ensemble import AdaBoostClassifier, AdaBoostRegressor 
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import roc_auc_score
from pathlib import Path
import datetime as dt
import time
import pytz
import json
import os

In [2]:
pst = pytz.timezone('America/Los_Angeles')
dt_str = dt.datetime.now(pst).strftime("%Y-%m-%d-%I_%M_%S_%p")
dt_str

'2026-08-10-07_35_47_PM'

In [3]:
oof_experiment_config = {
     "experiment": {
        "model": "xgboost",
        "type": "baseline",
        "dataset_type": "features_ohe_imputed_scaled",
        "description": "xgboost + ohe features + imputation + scaled",
        "target": "classification",
    },

    "cv": {
        "strategy": "StratifiedKFold",
        "n_splits": 5,
        "shuffle": True,
        "random_state": 0
    },

    "params": {
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "n_estimators": 1100,
        "learning_rate": 0.1,
        "max_depth": 5,
        "tree_method": "hist",
        "enable_categorical": True,
        "early_stopping_rounds": 10,
        "device": "cuda"
    },
    
    "fit_params": {
    }
}
oof_experiment_config["experiment"]["id"] = f"{dt_str}_{oof_experiment_config["experiment"]["model"]}"
experiment_config = oof_experiment_config.copy()

In [4]:
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    data_path = "/kaggle/input/datasets/abhinavneelam/smartphone-addiction/data"
    output_path = "/kaggle/working/"
else:
    data_path = "../../data"
    output_path = "../../"

In [5]:
ss = pd.read_csv(f"{data_path}/raw/sample_submission.csv")
target_column = ss.columns[-1]
target_column

'addicted_label'

In [6]:
raw_train_id = pd.read_csv(f"{data_path}/raw/train.csv")["id"]
X = pd.read_csv(f"{data_path}/processed/train_{experiment_config["experiment"]["dataset_type"]}.csv")
X_test = pd.read_csv(f"{data_path}/processed/test_{experiment_config["experiment"]["dataset_type"]}.csv")
y = pd.read_csv(f"{data_path}/processed/train_labels.csv")

X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 25 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   age                               691369 non-null  float64
 1   daily_screen_time_hours           691369 non-null  float64
 2   social_media_hours                691369 non-null  float64
 3   gaming_hours                      691369 non-null  float64
 4   work_study_hours                  691369 non-null  float64
 5   sleep_hours                       691369 non-null  float64
 6   notifications_per_day             691369 non-null  float64
 7   app_opens_per_day                 691369 non-null  float64
 8   weekend_screen_time               691369 non-null  float64
 9   stress_level                      691369 non-null  float64
 10  academic_work_impact              691369 non-null  float64
 11  daily_free_hours                  691369 non-null  f

In [7]:
cat_cols = X.select_dtypes(include=["object", "string"]).columns

for frame in [X, X_test]:
    for col in cat_cols:
        if experiment_config["experiment"]["model"] == "catboost":
            frame[col] = frame[col].fillna('Missing')
        
        frame[col] = frame[col].astype('category')

In [8]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 25 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   age                               691369 non-null  float64
 1   daily_screen_time_hours           691369 non-null  float64
 2   social_media_hours                691369 non-null  float64
 3   gaming_hours                      691369 non-null  float64
 4   work_study_hours                  691369 non-null  float64
 5   sleep_hours                       691369 non-null  float64
 6   notifications_per_day             691369 non-null  float64
 7   app_opens_per_day                 691369 non-null  float64
 8   weekend_screen_time               691369 non-null  float64
 9   stress_level                      691369 non-null  float64
 10  academic_work_impact              691369 non-null  float64
 11  daily_free_hours                  691369 non-null  f

In [9]:
def make_model(config):
    name = config["experiment"]["model"]
    target = config["experiment"]["target"]
    params = config["params"]

    model_dict = {
        "adaboost": (AdaBoostClassifier, AdaBoostRegressor),
        "gradientboost": (GradientBoostingClassifier, GradientBoostingRegressor),
        "catboost": (CatBoostClassifier, CatBoostRegressor),
        "xgboost": (XGBClassifier, XGBRegressor),
        "lightgbm": (LGBMClassifier, LGBMRegressor),
        "randomforest": (RandomForestClassifier, RandomForestRegressor),
        "extratrees": (ExtraTreesClassifier, ExtraTreesRegressor),
        "hgbc": (HistGradientBoostingClassifier, HistGradientBoostingRegressor),
        "knn": (KNeighborsClassifier, KNeighborsRegressor),
        "sgd": (SGDClassifier, SGDRegressor),
        "linear": (LogisticRegression, LinearRegression),
        "decisiontree": (DecisionTreeClassifier, DecisionTreeRegressor),
        "mlp": (MLPClassifier, MLPRegressor),
    }

    target_index = target == "regression"
    model_class = model_dict[name][target_index]

    return model_class(**params)

In [10]:
def oof_fit(config, model, X_train, y_train, X_valid, y_valid):
    name = config["experiment"]["model"]
    no_eval_models = ["adaboost", "gradientboost", "randomforest", "extratrees", "knn", "sgd", "linear", "decisiontree", "mlp"]
    fit_params = config["fit_params"]

    if name in no_eval_models:
        model.fit(X_train, y_train)
    elif name == "hgbc":
        model.fit(X_train, y_train, X_val=X_valid, y_val=y_valid)
    elif name == "lightgbm":
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], **fit_params)
    else:
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)])

In [11]:
cv_config = experiment_config["cv"]
kf = StratifiedKFold(n_splits=cv_config["n_splits"], random_state=cv_config["random_state"], shuffle=cv_config["shuffle"])

y_cv = pd.Series(index=y.index, dtype=float, name=target_column)
fold_scores = []

start_time = time.time()

for train_index, valid_index in kf.split(X, y):
    X_train, X_valid = X.iloc[train_index], X.iloc[valid_index]
    y_train, y_valid = y.iloc[train_index], y.iloc[valid_index]

    model = make_model(oof_experiment_config)
    oof_fit(oof_experiment_config, model, X_train, y_train, X_valid, y_valid)

    y_pred = model.predict_proba(X_valid)[:, 1]

    y_cv.iloc[valid_index] = y_pred

    fold_auc_score = roc_auc_score(y_valid, y_pred)
    fold_scores.append(round(fold_auc_score, 5))

elapsed = time.time() - start_time

y_pred_df = pd.concat([raw_train_id, y_cv], axis=1)

[0]	validation_0-auc:0.91877
[1]	validation_0-auc:0.92323
[2]	validation_0-auc:0.92666
[3]	validation_0-auc:0.92897
[4]	validation_0-auc:0.92880
[5]	validation_0-auc:0.92861
[6]	validation_0-auc:0.92829
[7]	validation_0-auc:0.92814
[8]	validation_0-auc:0.92841
[9]	validation_0-auc:0.92858
[10]	validation_0-auc:0.92872
[11]	validation_0-auc:0.92849
[12]	validation_0-auc:0.92940
[13]	validation_0-auc:0.92947
[14]	validation_0-auc:0.92958
[15]	validation_0-auc:0.93015
[16]	validation_0-auc:0.93035
[17]	validation_0-auc:0.93075
[18]	validation_0-auc:0.93094
[19]	validation_0-auc:0.93130
[20]	validation_0-auc:0.93142
[21]	validation_0-auc:0.93172
[22]	validation_0-auc:0.93181
[23]	validation_0-auc:0.93217
[24]	validation_0-auc:0.93263
[25]	validation_0-auc:0.93301
[26]	validation_0-auc:0.93327
[27]	validation_0-auc:0.93354
[28]	validation_0-auc:0.93377
[29]	validation_0-auc:0.93392
[30]	validation_0-auc:0.93422
[31]	validation_0-auc:0.93444
[32]	validation_0-auc:0.93457
[33]	validation_0-au

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [02:36:00] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


[0]	validation_0-auc:0.91938
[1]	validation_0-auc:0.92246
[2]	validation_0-auc:0.92528
[3]	validation_0-auc:0.92691
[4]	validation_0-auc:0.92724
[5]	validation_0-auc:0.92703
[6]	validation_0-auc:0.92685
[7]	validation_0-auc:0.92692
[8]	validation_0-auc:0.92661
[9]	validation_0-auc:0.92684
[10]	validation_0-auc:0.92686
[11]	validation_0-auc:0.92765
[12]	validation_0-auc:0.92788
[13]	validation_0-auc:0.92869
[14]	validation_0-auc:0.92876
[15]	validation_0-auc:0.92875
[16]	validation_0-auc:0.92911
[17]	validation_0-auc:0.92955
[18]	validation_0-auc:0.92973
[19]	validation_0-auc:0.93015
[20]	validation_0-auc:0.93028
[21]	validation_0-auc:0.93058
[22]	validation_0-auc:0.93070
[23]	validation_0-auc:0.93095
[24]	validation_0-auc:0.93129
[25]	validation_0-auc:0.93152
[26]	validation_0-auc:0.93165
[27]	validation_0-auc:0.93196
[28]	validation_0-auc:0.93228
[29]	validation_0-auc:0.93248
[30]	validation_0-auc:0.93267
[31]	validation_0-auc:0.93291
[32]	validation_0-auc:0.93310
[33]	validation_0-au

In [12]:
valid_auc_score = roc_auc_score(y, y_cv)

print("Validation AUC:", valid_auc_score)

Validation AUC: 0.9624915924982934


In [13]:
metrics = {
    "experiment": dt_str + f"_{experiment_config["experiment"]["model"]}",
    "model": f"{experiment_config["experiment"]["model"]}",
    "dataset_type": f"{experiment_config["experiment"]["dataset_type"]}",
    "cv": {
        "strategy": f"{experiment_config["cv"]["strategy"]}",
        "n_splits": experiment_config["cv"]["n_splits"],
        "random_state": experiment_config["cv"]["random_state"],
        "fold_scores": fold_scores,
        "mean": round(sum(fold_scores) / len(fold_scores), 5),
        "std": round(float(pd.Series(fold_scores).std(ddof=1)), 5)
    },
    "primary_metric": {
        "name": "auc",
        "value": round(valid_auc_score, 5)
    },
    "training": {
        "duration_seconds": round(elapsed, 2)
    }
}

In [14]:
params = experiment_config["params"]
tree_param_keys = ["n_estimators", "max_iter"]

best_iter = None
if hasattr(model, "best_iteration_"):
    best_iter = model.best_iteration_
elif hasattr(model, "best_iteration"):
    best_iter = model.best_iteration
elif hasattr(model, "n_iter_"):
    n_iter_val = model.n_iter_
    
    if isinstance(n_iter_val, np.ndarray):
        if n_iter_val.size == 1:
            best_iter = n_iter_val.item()
        else:
            best_iter = int(n_iter_val.max())
    else:
        best_iter = int(n_iter_val)

if best_iter is not None:
    for key in tree_param_keys:
        if key in params:
            params[key] = best_iter

training_only_params = [
    "early_stopping_rounds",
    "early_stopping",
    "n_iter_no_change",
    "validation_fraction",
    "eval_set",
]

for key in training_only_params:
    if key in params:
        del params[key]

In [15]:
model = make_model(experiment_config)
model.fit(X, y)

y_pred = model.predict_proba(X_test)[:, 1]
ss[target_column] = y_pred
ss

,id,addicted_label
0,691369,0.999635
1,691370,0.948989
2,691371,0.971056
3,691372,0.991785
4,691373,0.999362
...,...,...
296297,987666,1.000000
296298,987667,0.904485
296299,987668,0.219474
296300,987669,0.740476


In [16]:
experiment_path = Path(output_path) / "experiments" / f"{dt_str}_{experiment_config["experiment"]["model"]}"
experiment_path.mkdir(parents=True, exist_ok=True)

with open(experiment_path / "oof_config.json", "w") as f:
    json.dump(oof_experiment_config, f, indent=4)

with open(experiment_path / "full_config.json", "w") as f:
    json.dump(experiment_config, f, indent=4)

with open(experiment_path / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

y_pred_df.to_csv(experiment_path / "oof.csv", index=False)
joblib.dump(model, experiment_path / f"{experiment_config["experiment"]["model"]}.pkl")
ss.to_csv(experiment_path / f"{experiment_config["experiment"]["model"]}_submission.csv", index=False)